## Vaccination in 2021 

In [6]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import matplotlib.dates as mdates
#print(smf)
import covasim as cv

print(cv.__file__)
print(hasattr(cv, "Sim"))
print(cv.Sim)
import optuna

Covasim 3.1.8 (2026-05-29) — © 2020-2026 by IDM
/storage/hcraddock/GitHub/covasim4wastewater/covasim/__init__.py
True
<class 'covasim.sim.Sim'>


- source: https://www.sandiegocounty.gov/content/dam/sdc/hhsa/programs/phs/documents/HSABDocuments/HSABDocuments2021/September2021/COVID_19_Update_HSAB_Meeting_09.07.2021.pdf

- source: https://www.countynewscenter.com/san-diego-county-covid-19-update-6-30-2021/

In [1]:
# --- County figures (source: county updates 30 Jun and 1 Sep 2021) ---
POP_COUNTY       = 3347827
POP_COUNTY_12P   = 2802581
POP_PL           = 2116170                      # Point Loma catchment (from cases.csv)

In [2]:
# County head counts, residents 12+
county_n_one_dose_0630 = 2190000   
county_n_full_0630   = 1870000
county_n_one_dose_0901 = 2407528
county_n_full_0901   = 2112936

In [3]:
# Share of ALL residents (Covasim needs everyone, not just 12+).
# ASSUMPTION: Point Loma coverage = county coverage (no zip-level data used yet)
pl_share_one_dose_0630 = county_n_one_dose_0630 / POP_COUNTY
print(f"Share of one dose 06/30: {pl_share_one_dose_0630:.3f}")

pl_share_full_0630     = county_n_full_0630     / POP_COUNTY
print(f"Share of fully vaccinated 06/30: {pl_share_full_0630:.3f}")

pl_share_one_dose_0901 = county_n_one_dose_0901 / POP_COUNTY
print(f"Share of one dose 09/01: {pl_share_one_dose_0901:.3f}")

pl_share_full_0901     = county_n_full_0901     / POP_COUNTY
print(f"Share of fully vaccinated 09/01: {pl_share_full_0901:.3f}")

Share of one dose 06/30: 0.654
Share of fully vaccinated 06/30: 0.559
Share of one dose 09/01: 0.719
Share of fully vaccinated 09/01: 0.631


In [4]:
# Implied Point Loma head counts on 30 June
pl_n_one_dose_0630 = pl_share_one_dose_0630 * POP_PL
print(f"One dose 06/30: {pl_n_one_dose_0630:,.0f}")
pl_n_full_0630     = pl_share_full_0630     * POP_PL
print(f"Fully vaccinated 06/30: {pl_n_full_0630:,.0f}")

One dose 06/30: 1,384,305
Fully vaccinated 06/30: 1,182,032


In [10]:
pl_share_one_dose_0630

0.6541556657497535

## Probability/Chance of being Vaccinated

- Chance of NOT being vaccinated on a given day: 1 - p
- Chance of NOT being vaccinated on any of 65 days in a row: (1 - p)^65 (multiply the same daily chance 65 times)
- Chance of being vaccinated at least once in 65 days: 1 - (1 - p)^65

In [7]:
# Vaccination continuing during the sim: 28 Jun -> 1 Sep (65 days), drawn from the not-yet-vaccinated
pl_vacc_days_in_sim = 65
pl_daily_vacc_prob = cv.historical_vaccinate_prob.estimate_prob(
    duration=pl_vacc_days_in_sim,
    coverage=(pl_share_one_dose_0901 - pl_share_one_dose_0630) / (1 - pl_share_one_dose_0630))
pl_vacc_during_sim = dict(days=np.arange(0, pl_vacc_days_in_sim), prob=float(pl_daily_vacc_prob))

print(f"one dose 30 Jun: {pl_share_one_dose_0630:.3f} -> {pl_n_one_dose_0630:,.0f} people")
print(f"fully vaccinated 30 Jun: {pl_share_full_0630:.3f} -> {pl_n_full_0630:,.0f} people")
print(f"one dose 1 Sep: {pl_share_one_dose_0901:.3f}, daily prob in sim: {pl_daily_vacc_prob:.4f}")

one dose 30 Jun: 0.654 -> 1,384,305 people
fully vaccinated 30 Jun: 0.559 -> 1,182,032 people
one dose 1 Sep: 0.719, daily prob in sim: 0.0032


In [11]:
# Uptake among the unvaccinated: fraction of people with no dose on 30 Jun who had a first dose by 1 Sep
#   numerator   = rise in the share of ALL residents with >=1 dose (0.719 - 0.654 = 0.065)
#   denominator = share still unvaccinated on 30 Jun (1 - 0.654 = 0.346)
pl_uptake_unvacc = (pl_share_one_dose_0901 - pl_share_one_dose_0630) / (1 - pl_share_one_dose_0630)
print(pl_uptake_unvacc)   # about 0.188

0.1878760816598682


In [12]:
round(pl_uptake_unvacc, 3)  # about 0.188   

0.188